## Thêm các thư viện cần thiết

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sweetviz as sv

d:\ml\VTI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_path = r'D:\DOAN\machine_learning\lab05\data\marketing_campaign.csv'
report_dir = r'D:\DOAN\machine_learning\lab05\Lab1_EDA\reports'
os.makedirs(report_dir, exist_ok=True)

In [3]:
df=pd.read_csv(data_path, sep='\t')

In [4]:
df.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   str    
 3   Marital_Status       2240 non-null   str    
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   str    
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases    2240 non-null   int64  
 16 

In [6]:
df.shape

(2240, 29)

* File marketing_campaign.csv ngăn cách các cột bằng dấu tab nên phải đọc với sep='\t'.

* Tập dữ liệu Customer Personality Analysis gồm thông tin khách hàng (năm sinh, học vấn, tình trạng hôn nhân, thu nhập, số con), chi tiêu theo nhóm sản phẩm (Mnt...), hành vi mua hàng (Num...) và phản hồi các chiến dịch (AcceptedCmp, Response).

## Kiểm tra chất lượng dữ liệu

In [7]:
cols=['Income']
def ismissing(cols, df: pd.DataFrame):
    for col in cols:
        missingcount = df[col].isnull().sum()
        totalcount = len(df[col])
        missingper = missingcount / totalcount * 100

        print(f"{col}:")
        print(f"  Missing count: {missingcount}")
        print(f"  Missing percentage: {missingper:.2f}%")

In [8]:
ismissing(cols,df)

Income:
  Missing count: 24
  Missing percentage: 1.07%


In [9]:
print(f'Số dòng trùng lặp: {df.duplicated().sum()}')
print(f'ID là duy nhất: {df["ID"].is_unique}')

Số dòng trùng lặp: 0
ID là duy nhất: True


In [10]:
df.nunique().sort_values()

Z_Revenue                 1
Z_CostContact             1
AcceptedCmp3              2
AcceptedCmp4              2
AcceptedCmp5              2
AcceptedCmp1              2
Complain                  2
AcceptedCmp2              2
Response                  2
Teenhome                  3
Kidhome                   3
Education                 5
Marital_Status            8
NumStorePurchases        14
NumCatalogPurchases      14
NumDealsPurchases        15
NumWebPurchases          15
NumWebVisitsMonth        16
Year_Birth               59
Recency                 100
MntFruits               158
MntSweetProducts        177
MntFishProducts         182
MntGoldProds            213
MntMeatProducts         558
Dt_Customer             663
MntWines                776
Income                 1974
ID                     2240
dtype: int64

In [11]:
df['Year_Birth'].describe()

count    2240.000000
mean     1968.805804
std        11.984069
min      1893.000000
25%      1959.000000
50%      1970.000000
75%      1977.000000
max      1996.000000
Name: Year_Birth, dtype: float64

* Chỉ cột Income bị thiếu (24 dòng, khoảng 1.07%), tỷ lệ rất nhỏ.

* Hai cột Z_CostContact và Z_Revenue chỉ có 1 giá trị duy nhất (hằng số), không mang thông tin nên sẽ bỏ qua khi tạo báo cáo. Cột ID chỉ là định danh nên cũng bỏ qua.

* Year_Birth có giá trị nhỏ bất thường (năm sinh quá xa), cần xem là ngoại lai hoặc nhập sai khi tiền xử lý.

In [12]:
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y')

## SweetViz

SweetViz là thư viện tạo báo cáo EDA tự động dưới dạng file HTML chỉ với vài dòng code. Các tính năng chính:

* **analyze()**: báo cáo cho một tập dữ liệu, gồm thống kê và biểu đồ từng biến (số lượng, thiếu, distinct, mean, median, độ lệch...) và ma trận mối quan hệ giữa các biến (associations).

* **target_feat**: chọn một biến mục tiêu, báo cáo sẽ hiển thị mối quan hệ của mọi biến với biến mục tiêu đó.

* **compare() / compare_intra()**: so sánh hai tập dữ liệu (train với test) hoặc so sánh hai nhóm trong cùng một tập dữ liệu.

* **FeatureConfig**: bỏ qua (skip) hoặc ép kiểu (force_num, force_cat, force_text) cho từng cột.

Cài đặt: pip install sweetviz

### 1. Báo cáo tổng quan toàn bộ dữ liệu

In [13]:
feature_config = sv.FeatureConfig(skip=['ID', 'Z_CostContact', 'Z_Revenue'])

report = sv.analyze(df, feat_cfg=feature_config)
report.show_html(filepath=os.path.join(report_dir, 'sweetviz_report.html'), open_browser=False)

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:02 -> (00:00 left)


Report D:\DOAN\machine_learning\lab05\Lab1_EDA\reports\sweetviz_report.html was generated.


### 2. Báo cáo với biến mục tiêu Response (khách hàng chấp nhận chiến dịch cuối)

In [14]:
report_target = sv.analyze(df, target_feat='Response', feat_cfg=feature_config)
report_target.show_html(filepath=os.path.join(report_dir, 'sweetviz_report_target_response.html'), open_browser=False)

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:02 -> (00:00 left)

Report D:\DOAN\machine_learning\lab05\Lab1_EDA\reports\sweetviz_report_target_response.html was generated.


### 3. So sánh hai nhóm khách hàng: có con nhỏ và không có con nhỏ

In [15]:
report_compare = sv.compare_intra(df, df['Kidhome'] == 0, ['Kidhome = 0', 'Kidhome > 0'], feat_cfg=feature_config)
report_compare.show_html(filepath=os.path.join(report_dir, 'sweetviz_report_compare_kidhome.html'), open_browser=False)

Feature: Kidhome                             |█▊        | [ 19%]   00:05 -> (00:24 left)

TypeError: 

Cannot convert series 'Kidhome' in COMPARED from its TYPE_CATEGORICAL
to the desired type TYPE_BOOL.
Check documentation for the possible coercion possibilities.
POSSIBLE RESOLUTIONS:
 -> Use the feat_cfg parameter (see docs on git) to force the column to be a specific type (may or may not help depending on the type)
 -> Modify the source data to be more explicitly of a single specific type
 -> This could also be caused by a feature type mismatch between source and compare dataframes:
    In that case, make sure the source and compared dataframes are compatible.


## Nhận xét

* Báo cáo tạo ra 3 file HTML trong thư mục reports, mở bằng trình duyệt để xem. Mỗi biến có histogram/bar chart và các chỉ số thống kê ngay bên cạnh, không cần viết code vẽ từng biểu đồ như các bài trước.

* Biến thu nhập (Income) và các biến chi tiêu (MntWines, MntMeatProducts...) lệch phải, tập trung ở mức thấp và có một số ít khách chi tiêu rất cao. Các biến chi tiêu thường tương quan dương với Income và nghịch với Kidhome.

* Khi chọn Response làm target, có thể thấy ngay những biến liên quan tới việc khách chấp nhận chiến dịch (ví dụ các cột AcceptedCmp, Recency, chi tiêu).

* Khi so sánh nhóm có con nhỏ và không có con nhỏ, nhóm không có con nhỏ thường có thu nhập và chi tiêu cao hơn.

* Ưu điểm: nhanh, trực quan, có so sánh nhóm và target. Hạn chế: ít tuỳ chỉnh biểu đồ, chạy chậm với dữ liệu lớn và có thể lỗi tương thích với phiên bản pandas/numpy mới (nếu gặp lỗi khi import hoặc analyze thì cài đúng phiên bản hoặc tạo môi trường ảo riêng).